In [7]:
import os
import sqlite3
import time
from datetime import datetime, timezone
import requests

THINGSPEAK_WRITE_API_KEY = os.environ.get("THINGSPEAK_WRITE_API_KEY")
DRY_RUN = os.environ.get("THINGSPEAK_DRY_RUN", "true").strip().lower() in {"1", "true", "yes", "on"}
THINGSPEAK_URL = "https://api.thingspeak.com/update"
DATABASE_PATH = "sensor_readings.sqlite3"
UPLOAD_INTERVAL_SECONDS = 20  # ThingSpeak free channels generally require >=15 s between updates
MAX_SAMPLES = int(os.environ.get("MAX_SAMPLES", "1"))  # Set to 0 for continuous collection

if MAX_SAMPLES < 0:
    raise ValueError("MAX_SAMPLES must be zero or a positive integer.")
if not DRY_RUN and not THINGSPEAK_WRITE_API_KEY:
    raise RuntimeError("Set THINGSPEAK_WRITE_API_KEY in the environment before running this cell.")

print(f"SQLite database: {os.path.abspath(DATABASE_PATH)}")
print("ThingSpeak uploads are disabled (dry run)." if DRY_RUN else "ThingSpeak uploads are enabled.")

SQLite database: c:\Users\iamwa\OneDrive\Desktop\CL_2\IIOT\sensor_readings.sqlite3
ThingSpeak uploads are disabled (dry run).


In [8]:
def connect_db():
    return sqlite3.connect(DATABASE_PATH)

with connect_db() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS readings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            recorded_at TEXT NOT NULL,
            temperature_c REAL NOT NULL,
            humidity_pct REAL NOT NULL,
            cloud_entry_id TEXT,
            upload_ok INTEGER NOT NULL,
            upload_error TEXT
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_readings_recorded_at ON readings(recorded_at)")
print("Database table is ready.")

Database table is ready.


In [9]:
def read_sensor():
    """Replace this body with actual sensor driver calls. Values here are a demo."""
    import random
    return {"temperature_c": round(22 + random.random() * 5, 2),
            "humidity_pct": round(40 + random.random() * 20, 2)}

def save_reading(reading, recorded_at, cloud_entry_id=None, upload_ok=False, upload_error=None):
    with connect_db() as conn:
        conn.execute(
            "INSERT INTO readings(recorded_at, temperature_c, humidity_pct, cloud_entry_id, upload_ok, upload_error) VALUES (?, ?, ?, ?, ?, ?)",
            (recorded_at, reading["temperature_c"], reading["humidity_pct"], cloud_entry_id, int(upload_ok), upload_error),
        )

def upload_to_thingspeak(reading):
    response = requests.post(
        THINGSPEAK_URL,
        data={"api_key": THINGSPEAK_WRITE_API_KEY,
              "field1": reading["temperature_c"],
              "field2": reading["humidity_pct"]},
        timeout=15,
    )
    response.raise_for_status()
    entry_id = response.text.strip()
    if not entry_id or entry_id == "0":
        raise RuntimeError("ThingSpeak did not accept the update (entry ID 0). Check the update interval and channel fields.")
    return entry_id

In [10]:
def capture_and_send():
    reading = read_sensor()
    recorded_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    if DRY_RUN:
        message = "ThingSpeak upload skipped (dry run)"
        save_reading(reading, recorded_at, upload_ok=False, upload_error=message)
        print(f"{recorded_at}: {reading} → {message}; saved locally")
        return False
    try:
        entry_id = upload_to_thingspeak(reading)
        save_reading(reading, recorded_at, cloud_entry_id=entry_id, upload_ok=True)
        print(f"{recorded_at}: {reading} → ThingSpeak entry {entry_id}; saved locally")
        return True
    except (requests.RequestException, RuntimeError) as exc:
        save_reading(reading, recorded_at, upload_ok=False, upload_error=str(exc))
        print(f"{recorded_at}: {reading} → cloud upload failed ({exc}); saved locally")
        return False

# Send one sample now.
capture_and_send()

2026-11-17T18:08:32+00:00: {'temperature_c': 24.68, 'humidity_pct': 49.89} → ThingSpeak upload skipped (dry run); saved locally


False

In [11]:
sample_count = 0
try:
    while MAX_SAMPLES == 0 or sample_count < MAX_SAMPLES:
        started = time.monotonic()
        capture_and_send()
        sample_count += 1
        if MAX_SAMPLES and sample_count >= MAX_SAMPLES:
            break
        remaining = UPLOAD_INTERVAL_SECONDS - (time.monotonic() - started)
        if remaining > 0:
            time.sleep(remaining)
except KeyboardInterrupt:
    print("Collection stopped.")

2026-11-17T18:08:32+00:00: {'temperature_c': 22.26, 'humidity_pct': 51.87} → ThingSpeak upload skipped (dry run); saved locally


In [12]:
with connect_db() as conn:
    rows = conn.execute(
        "SELECT id, recorded_at, temperature_c, humidity_pct, cloud_entry_id, upload_ok, upload_error FROM readings ORDER BY id DESC LIMIT 10"
    ).fetchall()
for row in rows:
    print(row)

(4, '2026-11-17T18:08:32+00:00', 22.26, 51.87, None, 0, 'ThingSpeak upload skipped (dry run)')
(3, '2026-11-17T18:08:32+00:00', 24.68, 49.89, None, 0, 'ThingSpeak upload skipped (dry run)')
(2, '2026-11-17T18:08:20+00:00', 22.86, 59.62, None, 0, 'ThingSpeak upload skipped (dry run)')
(1, '2026-11-17T18:08:17+00:00', 25.0, 48.83, None, 0, 'ThingSpeak upload skipped (dry run)')
